# Démo 2 · Exploration et visualisation des données

Ce notebook couvre **l’exploration des données** ainsi que **la visualisation et la science des données exploratoire**.
Examinez les événements bruts de la NHL avec un curseur, rejouez les tirs sur une patinoire et
utilisez un graphique interactif des tirs cumulés pour discuter du déroulement du match.

Commencez par [D’une API à une table pandas](01_data_acquisition_and_cleaning.ipynb)
pour la leçon d’acquisition et de nettoyage. Cette suite fonctionne dans un noyau vierge :
la courte préparation ci-dessous charge le même match et reconstruit la même table de tirs.



## Configuration : choisir l’environnement local ou Colab

**En local :** lancez `uv sync` à la racine du dépôt `ift3700-6758`, puis sélectionnez le noyau de l’environnement commun `.venv` sous Python 3.11. Consultez le [README du dépôt](../../../README.md) pour la configuration. Sautez la cellule d’installation facultative : elle n’installe rien en local.

**Google Colab | sans clonage du dépôt ni installation manuelle de uv :**

1. Ouvrez ce notebook dans Colab et choisissez un runtime **CPU**. L’installation nécessite Internet.
2. Dans la **cellule facultative ci-dessous**, définissez `INSTALL_COLAB_PACKAGES = True` et exécutez-la une fois. La cellule installe `uv`, puis l’utilise pour installer les bibliothèques dans le Python actuel du notebook. `sys.executable` est le chemin de ce Python; `subprocess.check_call` lance une commande et s’arrête si elle échoue.
3. Si Colab demande un redémarrage après l’installation, choisissez **Runtime → Restart session**.
4. Remettez le paramètre à `False`, puis exécutez les cellules de haut en bas avec **Maj+Entrée**. Répétez l’installation lorsque Colab vous attribue un nouveau runtime; un simple redémarrage de session conserve les paquets installés.

Ce notebook utilise **requests** pour les requêtes HTTP et **pandas** pour les tables; `json` et `pathlib` sont inclus dans Python. **Plotly** produit les graphiques interactifs et **ipywidgets** fournit les curseurs. La cellule active aussi les widgets dans Colab, même lorsque l’installation est désactivée.

La première exécution nécessite Internet pour télécharger le match. Les suivantes réutilisent le fichier JSON sauvegardé.


In [ ]:
# OPTIONAL: run once in a fresh Colab runtime; skip during local development.
INSTALL_COLAB_PACKAGES = False  # Set to True to install; reset to False afterward.

import sys
import subprocess

# Detect Colab; otherwise use the local .venv.
try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not IN_COLAB:
    print("Local environment: skipped. Use uv sync in your terminal.")
elif not INSTALL_COLAB_PACKAGES:
    print("Installation skipped. Set INSTALL_COLAB_PACKAGES = True if this is a fresh Colab runtime.")
else:
    # Install into the Python running these cells.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "uv>=0.8,<1"])
    packages = [
        "pandas>=2.2,<4",
        "requests>=2.31,<3",
        "plotly>=6,<7",
        "ipywidgets>=8.1,<9",
    ]
    subprocess.check_call([sys.executable, "-m", "uv", "pip", "install",
                           "--python", sys.executable, *packages])
    print("Installation finished. Restart the session if Colab requests it.")

if IN_COLAB:
    # Enable interactive sliders, including after a session restart.
    output.enable_custom_widget_manager()


### Le dossier de données
`Path` construit les chemins. Dans Colab, on utilise le dossier courant.
En local, on retrouve la racine du dépôt pour réutiliser le même cache.
Le JSON est dans `data/raw`; les exports vont dans `data/processed`.

In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not IN_COLAB:
    # Walk up to the local repository.
    for folder in [ROOT, *ROOT.parents]:
        if (folder / "demo_2").is_dir():
            ROOT = folder
            break
os.chdir(ROOT)
print(ROOT / "data/raw/2025030311.json")

## Préparer les données pour l’exploration

Ce bref rappel de la partie 1 utilise le même match et les mêmes règles de nettoyage.
Chargez le JSON brut en cache, ou téléchargez-le une seule fois s’il est absent;
aplatissez les événements, gardez les tirs et buts, puis vérifiez leurs identifiants.
Aucune variable d’un autre notebook n’est nécessaire. Le notebook d’acquisition
d’origine reste inchangé.


In [ ]:
import json
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

game_id = 2025030311
url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/play-by-play"
print(url)

raw_dir = Path("data/raw")
raw_dir.mkdir(parents=True, exist_ok=True)
raw_path = raw_dir / f"{game_id}.json"
print("Raw data file:", raw_path.resolve())

if raw_path.exists():
    print("Already downloaded — reusing", raw_path.name)
else:
    response = requests.get(url, timeout=30)
    print("HTTP status:", response.status_code)
    print("Content type:", response.headers.get("Content-Type"))
    response.raise_for_status()
    downloaded_game = response.json()

    # Check we received the requested game's event data before saving it.
    assert downloaded_game["id"] == game_id
    assert isinstance(downloaded_game["plays"], list)
    raw_path.write_text(json.dumps(downloaded_game, indent=2), encoding="utf-8")
    print("Saved", raw_path.name)

game = json.loads(raw_path.read_text(encoding="utf-8"))
assert game["id"] == game_id
print("Python type:", type(game))
print("Top-level keys:", list(game.keys()))

In [ ]:
plays = game["plays"]
events = pd.json_normalize(plays)
print("Rows, columns:", events.shape)
print("Columns:", events.columns.tolist())
display(events.head())

keep = events["typeDescKey"].isin(["shot-on-goal", "goal"])
shots = events.loc[keep].copy()
print("All events:", len(events))
print("Retained shots and goals:", len(shots))
print("Other events left out:", len(events) - len(shots))

column_names = {
    "eventId": "event_id",
    "periodDescriptor.number": "period",
    "periodDescriptor.periodType": "period_type",
    "timeInPeriod": "time_in_period",
    "typeDescKey": "event_type",
    "details.eventOwnerTeamId": "team_id",
    "details.xCoord": "x",
    "details.yCoord": "y",
    "details.shotType": "shot_type",
}
shots = shots.reindex(columns=list(column_names)).rename(columns=column_names)
shots["game_id"] = game["id"]
shots["season"] = game["season"]
shots["game_type"] = game["gameType"]
shots["is_goal"] = shots["event_type"].eq("goal")

team_names = {
    game["awayTeam"]["id"]: game["awayTeam"]["abbrev"],
    game["homeTeam"]["id"]: game["homeTeam"]["abbrev"],
}
shots["team"] = shots["team_id"].map(team_names)
shots = shots.convert_dtypes()
display(shots.head(10))

assert len(events) == len(plays), "Flattening changed the number of events."
assert len(shots) == int(keep.sum()), "Cleaning changed the number of retained events."
assert shots[["game_id", "event_id"]].notna().all().all(), "Missing event identifier."
assert not shots.duplicated(["game_id", "event_id"]).any(), "Duplicate event identifiers."

print("Checks passed.")
display(shots.isna().sum().rename("missing_count").to_frame())

## Objectif 2 · examiner avant de tracer
`events` contient tous les événements; `shots` contient les tirs au but **buts inclus**,
préparés avec les règles de nettoyage de la partie 1. Commencez par compter les types d’événements.
Que représente une ligne ?



In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from ipywidgets import interact, IntSlider

LANG = 'en'
pio.renderers.default = "colab" if IN_COLAB else "plotly_mimetype+notebook"
teams = [game["awayTeam"]["abbrev"], game["homeTeam"]["abbrev"]]
colors = dict(zip(teams, ["#2364ce", "#e25c45"]))

display(events["typeDescKey"].value_counts().rename("count").to_frame())
if LANG == "fr":
    print(f"{len(events)} événements → {len(shots)} tirs cadrés, buts inclus")
else:
    print(f"{len(events)} events → {len(shots)} shots on goal, goals included")

### Une petite fonction réutilisable : dessiner la patinoire
Une fonction regroupe des instructions. Celle-ci ajoute un rectangle et trois lignes
à une figure Plotly, puis la renvoie. Les axes fixes conservent les proportions.
Les coordonnées sont en pieds. Ce dessin est un schéma, pas un paquet spécialisé hockey.

In [ ]:
def draw_rink(fig):
    # A 200-foot by 85-foot rectangle.
    fig.add_shape(type="rect", x0=-100, x1=100, y0=-42.5, y1=42.5,
                  line_color="#7f95aa")
    for x, colour in [(-25, "#9bc2e6"), (0, "#e6a0a0"), (25, "#9bc2e6")]:
        fig.add_vline(x=x, line_color=colour)
    fig.update_xaxes(range=[-103, 103], title="x (ft)", constrain="domain")
    fig.update_yaxes(range=[-46, 46], title="y (ft)", scaleanchor="x", scaleratio=1)
    fig.update_layout(template="plotly_white", height=460,
                      margin=dict(l=55, r=25, t=55, b=50))
    return fig

### Un curseur devient l'argument d'une fonction
`@interact` appelle la fonction à chaque changement de curseur.
`i` est une **position dans la liste**, pas l'identifiant NHL de l'événement.
Le tableau ci-dessous donne les positions des buts et débuts de période à essayer.
Sans coordonnées, on affiche le JSON et une patinoire vide, jamais un faux point `(0, 0)`.

In [ ]:
examples = events["typeDescKey"].isin(["goal", "period-start"])
display(events.loc[examples, ["eventId", "typeDescKey"]])

@interact(i=IntSlider(min=0, max=len(game["plays"])-1, continuous_update=False))
def inspect_event(i):
    play = game["plays"][i]
    print(json.dumps(play, indent=2, ensure_ascii=False))
    details = play.get("details", {})
    x = details.get("xCoord")
    y = details.get("yCoord")
    fig = go.Figure()
    # Plot only if both coordinates exist.
    if x is not None and y is not None:
        fig.add_scatter(x=[x], y=[y], mode="markers", marker_size=18)
    fig.update_layout(title=f"eventId={play['eventId']} · {play['typeDescKey']}")
    draw_rink(fig).show()

## Objectif 3 · construire une horloge
`mm:ss` est le temps écoulé dans la période. Pour les périodes 1–3 :
`minute = (period - 1) × 20 + mm + ss / 60`.
Une copie nommée `timed` exclut les prolongations et les tirs de barrage; `shots` reste intacte.


In [ ]:
regular = (shots["period_type"] == "REG") & shots["period"].between(1, 3)
timed = shots.loc[regular].copy()
clock = timed["time_in_period"].str.split(":", expand=True).astype(float)
timed["minute"] = (timed["period"] - 1) * 20 + clock[0] + clock[1] / 60
timed = timed.sort_values(["minute", "event_id"])
label = "Tirs exclus du replay :" if LANG == "fr" else "Shots excluded from replay:"
print(label, len(shots) - len(timed))
display(timed[["period", "time_in_period", "minute", "team"]].head())

### Le match en une minute
Le bouton ▶ avance le curseur. À chaque minute, filtrez les tirs qui ont déjà eu lieu.
`color` indique l’équipe; `symbol` distingue les buts (étoiles).
Arrêtez à la minute 20 et prédisez la suite. Ce n’est pas un suivi de la rondelle.


In [ ]:
import ipywidgets as widgets

# Link the play button to the slider.
play = widgets.Play(min=0, max=60, interval=500)
minute_slider = IntSlider(min=0, max=60, continuous_update=False)
widgets.jslink((play, "value"), (minute_slider, "value"))
display(play)

@interact(minute=minute_slider)
def replay_at(minute):
    visible = timed.loc[timed["minute"] <= minute].dropna(subset=["x", "y"])
    title = f"Minute {minute} · {len(visible)} tirs" if LANG == "fr" else f"Minute {minute} · {len(visible)} shots"
    fig = px.scatter(visible, x="x", y="y", color="team", symbol="is_goal",
                     color_discrete_map=colors, symbol_map={False: "circle", True: "star"},
                     category_orders={"team": teams},
                     hover_data=["period", "time_in_period"], title=title)
    fig.update_traces(marker_size=13, marker_opacity=0.85)
    draw_rink(fig).show()

### De l’animation à l’analyse : les tirs cumulés
`cumcount() + 1` numérote les tirs de chaque équipe dans l’ordre chronologique.
Une courbe en escalier ne suggère pas de tirs entre les événements.
Une forte pente signifie beaucoup de tirs, pas une forte possession.


In [ ]:
timed["cumulative_shots"] = timed.groupby("team").cumcount() + 1
fig = go.Figure()
for team, group in timed.groupby("team"):
    # Include the start and end of the game.
    minutes = [0] + group["minute"].tolist() + [60]
    totals = [0] + group["cumulative_shots"].tolist() + [len(group)]
    fig.add_scatter(x=minutes, y=totals, name=team, mode="lines",
                    line=dict(shape="hv", color=colors[team]))
    goals = group.loc[group["is_goal"]]
    fig.add_scatter(x=goals["minute"], y=goals["cumulative_shots"],
                    mode="markers", showlegend=False,
                    marker=dict(symbol="star", size=14, color=colors[team]),
                    hovertemplate=team + " · goal<extra></extra>")
for minute in [20, 40]:
    fig.add_vline(x=minute, line_dash="dot")
y_title = "Tirs cadrés cumulés" if LANG == "fr" else "Cumulative shots on goal"
title = "Quand les tirs s'accumulent-ils ?" if LANG == "fr" else "When do shots accumulate?"
fig.update_layout(title=title, xaxis_title="Minutes", yaxis_title=y_title,
                  template="plotly_white", hovermode="x unified")
fig.show()

### À vous · 3 minutes
Compter les tirs entre les minutes 15 et 20, par équipe.

In [ ]:
window = timed.loc[timed["minute"].between(15, 20)]
# Complete here.

<details><summary>Réponse de référence</summary>

```python
display(window.groupby("team").size())
```

</details>

## Export et conclusion
`fig` est la dernière figure principale. Son HTML inclut Plotly et s’ouvre sans noyau Python.
Les widgets ont besoin d’un noyau actif. Dans Colab, téléchargez l’export via le panneau Fichiers.
Un match illustre la méthode; le jalon demande plusieurs matchs/saisons et des cartes d’excès par heure.
**Cours 6 :** diapositives 5 (explorer/présenter), 9 (la question d’abord), 20–21 (une question par figure), 47 (interactivité et incertitude).


In [ ]:
output_dir = ROOT / "data/processed"
output_dir.mkdir(parents=True, exist_ok=True)
target = output_dir / "figure_b_en.html"
fig.write_html(target, include_plotlyjs=True)
print(target)